<a href="https://colab.research.google.com/github/Scadelai/PCD/blob/main/Copy_of_ProjetoPCD_kmeans.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Projeto PCD: K-means 1D (naive) com Paralelização Progressiva**

## Profs. Álvaro e Denise (Turmas I e N)

O algoritmo K-Means realiza uma operação de agrupamento ("clusterização") para mineração de dados, ou seja, permite agrupar amostras de um dado conjunto em grupos homogêneos. O método particiona um conjunto de *n* observações (pontos) em *k* grupos, onde cada ponto será associado ao grupo cuja média seja a mais próxima. A distância Euclidiana é geralmente a métrica adotada para medir a proximidade.
A "clusterização" é um problema *NP-hard*, mas existem algoritmos heurísticos eficientes que podem rapidamente encontrar um ótimo local. Nesta implementação, a aplicação recebe como entrada as coordenadas (1D) de *k* centróides iniciais e um conjunto de dados. O K-Means realiza um processo iterativo, no qual os pontos são reagrupados de acordo com a menor distância Euclidiana entre eles e os centróides. Em seguida, o centróide de cada partição é recalculado tomando a média de todos os pontos da partição, e todo o procedimento é repetido até que nenhum centróide seja alterado e nenhum ponto seja atribuído a outro grupo. Ao final, o algoritmo retorna as coordenadas dos *k* centróides finais.


## Objetivo

Implementar o **k-means em 1 dimensão** (pontos `X[i]` e centróides `C[c]`), medir **SSE** e **desempenho**, e **paralelizar** o núcleo do algoritmo em três etapas independentes:

1. **OpenMP (CPU memória compartilhada)**
2. **CUDA (GPU)**
3. **MPI (memória distribuída)**

## Entradas e saídas

* **Entradas (CSV, 1 coluna, sem cabeçalho):**

  * `dados.csv` com **N** valores (pontos).
  * `centroides_iniciais.csv` com **K** valores.
* **Saídas:**

  * No terminal: **iterações**, **SSE (*Sum of Squared Errors*, ou Soma dos Erros Quadráticos, em português) final**, **tempo total** (ms).
  * Arquivos: `assign.csv` (N linhas, índice do cluster por ponto) e `centroids.csv` (K linhas com centróides finais).

## Algoritmo (base “naive”)

Itere até `max_iter` ou até variar pouco o SSE (`eps`):

1. **Assignment:** para cada ponto, escolher o centróide mais próximo (minimiza $(x_i - c)^2$); acumular **SSE**.
2. **Update:** para cada cluster, **média** dos pontos atribuídos. Se um cluster ficar vazio, **copie `X[0]`** (estratégia simples).

---

## Etapa 0 — Versão sequencial (baseline)

* Executar a versão **sequencial** (fornecida mais abaixo).
* Coletar: **SSE por iteração**, **tempo total**, **iterações**.
* Salvar esses números: serão a **linha de base** para speedup.

---

## Etapa 1 — OpenMP (CPU)

**Meta:** paralelizar as funções de *assignment* e *update* na CPU.

### O que paralelizar

* **Assignment:** laço `for (i=0; i<N; ++i)`.
* **Update:**

  * opção A (mais simples): usar **acumuladores por thread** (`sum_thread[c]`, `cnt_thread[c]`) e **reduzir** após a região paralela;
  * opção B: usar `#pragma omp critical` e verificar impactos no desempenho.

### Medições

* **Escalonamento em threads:** T ∈ {1, 2, 4, 8, 16, …}.
* **Speedup** = tempo_serial / tempo_OpenMP.
* **Afinar:** *schedule* (`static` vs `dynamic`) e *chunk size*.
* **Validação:** SSE não deve **aumentar** ao longo das iterações (pode ficar igual se convergiu).

### Dica de compilação

```bash
gcc -O2 -fopenmp -std=c99 kmeans_1d_omp.c -o kmeans_1d_omp -lm
```

---

## Etapa 2 — CUDA (GPU)

**Meta:** mover o **assignment** para a GPU; o **update** pode ser feito na GPU (com atomics) ou no host (copiando `assign`).

### Desenho mínimo

* **Kernel de assignment:** 1 *thread* por ponto `i`.

  * Cada thread varre **K** centróides, calcula `d = (X[i]-C[c])^2`, guarda o melhor e escreve `assign[i]`.
  * (Opcional) carregar `C` em **memória constante**.
* **SSE:** reduzir no host somando os erros por ponto (ou fazer redução em blocos).
* **Update:**

  * opção A (mais simples): copiar `assign` para CPU e calcular médias no host;
  * opção B: usar **atomics** em `sum[c]` e `cnt[c]` na GPU e depois dividir.

### Medições

* **Tamanho de bloco** (p.ex., 128, 256, 512) × **grid**;
* **Tempos**: H2D/D2H, *kernel*, total;
* **Throughput**: pontos/s; **speedup** vs. serial e vs. OpenMP.

### Dica de compilação

```bash
nvcc -O2 kmeans_1d_cuda.cu -o kmeans_1d_cuda
```

---

## Etapa 3 — MPI (distribuída)

**Meta:** distribuir os **N** pontos entre **P** processos; centróides são **globais** a cada iteração.

### Passos por iteração

1. **Broadcast** (ou inicialização compartilhada): todos os processos têm `C`.
2. **Assignment local:** cada processo calcula `assign_local` e `SSE_local` para seu bloco de pontos.
3. **Redução global:**

   * somar `SSE_local` → `SSE_global` com `MPI_Reduce`;
   * somar `sum_local[c]` e `cnt_local[c]` para todos os clusters com `MPI_Allreduce`;
   * cada processo atualiza `C` com os resultados globais.
4. Próxima iteração até convergir.

### Medições

* **Strong scaling:** P ∈ {1, 2, 4, 8, …}.
* **Tempo de comunicação:** destacar o custo de `Allreduce`.
* **Speedup** vs. serial e OpenMP.

### Dica de compilação/execução

```bash
mpicc -O2 -std=c99 kmeans_1d_mpi.c -o kmeans_1d_mpi -lm
mpirun -np 4 ./kmeans_1d_mpi dados.csv centroides_iniciais.csv [args...]
```

---

## Conjuntos de teste sugeridos (1D)

* **Pequeno:** N=10^4, K=4
* **Médio:** N=10^5, K=8
* **Grande:** N=10^6, K=16 (se houver memória)
  Gere dados com mistura de faixas (ex.: perto de 0, 10, 20, 30) para facilitar a verificação visual.

---

## O que entregar

1. **Código no github**: `serial/`, `openmp/`, `cuda/`, `mpi/` (cada pasta com `README.md` de como compilar/rodar).
2. **Relatório curto (4–6 págs po etapa)**:

   * Ambiente (CPU/GPU/RAM/rede; versões de compilador).
   * Gráficos: **tempo**, **speedup**, **pontos/s** por etapa.
   * Para MPI: curva de **speedup** e comentário sobre custo de `Allreduce`.
   * Para CUDA: impacto de **block size** e custo de **transferência**.
   * Para OpenMP: efeito de **nº de threads** e de *schedule*.
   * Seções de **validação** (SSE por iteração, convergência, igualdade de resultados entre versões dentro de tolerância).
   * Análise de resultados e conclusões
   * Referências bibliográficas: apresente uma pequena revisão bibliográfica e compare seus resultados com outros encontrados na literatura.

---

## Critérios de avaliação

* **Desempenho e análise** (speedup, tempos de execução, eficiência e gargalos por arquitetura): **30%**
* **Corretude e reprodutibilidade** (SSE consistente, convergência, demonstração da corretude da execução): **30%**
* **Relatório** (clareza, gráficos, referências bibliográficas, análises e conclusões): **20%**
* **Qualidade do código e organização**: **10%**
* **Extra** (implementação e/ou análises não sugeridas no enunciado e que melhorem a qualidade científica do trabalho): **10%**


---

## Dicas rápidas

* Padronize **parâmetros** (N, K, `max_iter`, `eps`) entre as versões para comparar.
* Fixe uma **semente** ao gerar dados (quando aplicável) para repetibilidade.



##Exemplo: código sequencial e arquivos de entrada e saída

In [27]:
%%writefile centroides_iniciais.csv

10
30
60
90


Overwriting centroides_iniciais.csv


In [28]:
%%writefile dados.csv

1
2
3
4
5
6
7
8
4.5
5.5
18
19
20
21
22
23
19.5
20.5
100
125

Overwriting dados.csv


In [29]:
%%writefile kmeans_1d_naive.c

/* kmeans_1d_naive.c
   K-means 1D (C99), implementação "naive":
   - Lê X (N linhas, 1 coluna) e C_init (K linhas, 1 coluna) de CSVs sem cabeçalho.
   - Itera assignment + update até max_iter ou variação relativa do SSE < eps.
   - Salva (opcional) assign (N linhas) e centróides finais (K linhas).

   Compilar: gcc -O2 -std=c99 kmeans_1d_naive.c -o kmeans_1d_naive -lm
   Uso:      ./kmeans_1d_naive dados.csv centroides_iniciais.csv [max_iter=50] [eps=1e-4] [assign.csv] [centroids.csv]
*/

#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <math.h>
#include <time.h>

/* ---------- util CSV 1D: cada linha tem 1 número ---------- */
static int count_rows(const char *path){
    FILE *f = fopen(path, "r");
    if(!f){ fprintf(stderr,"Erro ao abrir %s\n", path); exit(1); }
    int rows=0; char line[8192];
    while(fgets(line,sizeof(line),f)){
        int only_ws=1;
        for(char *p=line; *p; p++){
            if(*p!=' ' && *p!='\t' && *p!='\n' && *p!='\r'){ only_ws=0; break; }
        }
        if(!only_ws) rows++;
    }
    fclose(f);
    return rows;
}

static double *read_csv_1col(const char *path, int *n_out){
    int R = count_rows(path);
    if(R<=0){ fprintf(stderr,"Arquivo vazio: %s\n", path); exit(1); }
    double *A = (double*)malloc((size_t)R * sizeof(double));
    if(!A){ fprintf(stderr,"Sem memoria para %d linhas\n", R); exit(1); }

    FILE *f = fopen(path, "r");
    if(!f){ fprintf(stderr,"Erro ao abrir %s\n", path); free(A); exit(1); }

    char line[8192];
    int r=0;
    while(fgets(line,sizeof(line),f)){
        int only_ws=1;
        for(char *p=line; *p; p++){
            if(*p!=' ' && *p!='\t' && *p!='\n' && *p!='\r'){ only_ws=0; break; }
        }
        if(only_ws) continue;

        /* aceita vírgula/ponto-e-vírgula/espaco/tab, pega o primeiro token numérico */
        const char *delim = ",; \t";
        char *tok = strtok(line, delim);
        if(!tok){ fprintf(stderr,"Linha %d sem valor em %s\n", r+1, path); free(A); fclose(f); exit(1); }
        A[r] = atof(tok);
        r++;
        if(r>R) break;
    }
    fclose(f);
    *n_out = R;
    return A;
}

static void write_assign_csv(const char *path, const int *assign, int N){
    if(!path) return;
    FILE *f = fopen(path, "w");
    if(!f){ fprintf(stderr,"Erro ao abrir %s para escrita\n", path); return; }
    for(int i=0;i<N;i++) fprintf(f, "%d\n", assign[i]);
    fclose(f);
}

static void write_centroids_csv(const char *path, const double *C, int K){
    if(!path) return;
    FILE *f = fopen(path, "w");
    if(!f){ fprintf(stderr,"Erro ao abrir %s para escrita\n", path); return; }
    for(int c=0;c<K;c++) fprintf(f, "%.6f\n", C[c]);
    fclose(f);
}

/* ---------- k-means 1D ---------- */
/* assignment: para cada X[i], encontra c com menor (X[i]-C[c])^2 */
static double assignment_step_1d(const double *X, const double *C, int *assign, int N, int K){
    double sse = 0.0;
    for(int i=0;i<N;i++){
        int best = -1;
        double bestd = 1e300;
        for(int c=0;c<K;c++){
            double diff = X[i] - C[c];
            double d = diff*diff;
            if(d < bestd){ bestd = d; best = c; }
        }
        assign[i] = best;
        sse += bestd;
    }
    return sse;
}

/* update: média dos pontos de cada cluster (1D)
   se cluster vazio, copia X[0] (estratégia naive) */
static void update_step_1d(const double *X, double *C, const int *assign, int N, int K){
    double *sum = (double*)calloc((size_t)K, sizeof(double));
    int *cnt = (int*)calloc((size_t)K, sizeof(int));
    if(!sum || !cnt){ fprintf(stderr,"Sem memoria no update\n"); exit(1); }

    for(int i=0;i<N;i++){
        int a = assign[i];
        cnt[a] += 1;
        sum[a] += X[i];
    }
    for(int c=0;c<K;c++){
        if(cnt[c] > 0) C[c] = sum[c] / (double)cnt[c];
        else           C[c] = X[0]; /* simples: cluster vazio recebe o primeiro ponto */
    }
    free(sum); free(cnt);
}

static void kmeans_1d(const double *X, double *C, int *assign,
                      int N, int K, int max_iter, double eps,
                      int *iters_out, double *sse_out)
{
    double prev_sse = 1e300;
    double sse = 0.0;
    int it;
    for(it=0; it<max_iter; it++){
        sse = assignment_step_1d(X, C, assign, N, K);
        /* parada por variação relativa do SSE */
        double rel = fabs(sse - prev_sse) / (prev_sse > 0.0 ? prev_sse : 1.0);
        if(rel < eps){ it++; break; }
        update_step_1d(X, C, assign, N, K);
        prev_sse = sse;
    }
    *iters_out = it;
    *sse_out = sse;
}

/* ---------- main ---------- */
int main(int argc, char **argv){
    if(argc < 3){
        printf("Uso: %s dados.csv centroides_iniciais.csv [max_iter=50] [eps=1e-4] [assign.csv] [centroids.csv]\n", argv[0]);
        printf("Obs: arquivos CSV com 1 coluna (1 valor por linha), sem cabeçalho.\n");
        return 1;
    }
    const char *pathX = argv[1];
    const char *pathC = argv[2];
    int max_iter = (argc>3)? atoi(argv[3]) : 50;
    double eps   = (argc>4)? atof(argv[4]) : 1e-4;
    const char *outAssign   = (argc>5)? argv[5] : NULL;
    const char *outCentroid = (argc>6)? argv[6] : NULL;

    if(max_iter <= 0 || eps <= 0.0){
        fprintf(stderr,"Parâmetros inválidos: max_iter>0 e eps>0\n");
        return 1;
    }

    int N=0, K=0;
    double *X = read_csv_1col(pathX, &N);
    double *C = read_csv_1col(pathC, &K);
    int *assign = (int*)malloc((size_t)N * sizeof(int));
    if(!assign){ fprintf(stderr,"Sem memoria para assign\n"); free(X); free(C); return 1; }

    clock_t t0 = clock();
    int iters = 0; double sse = 0.0;
    kmeans_1d(X, C, assign, N, K, max_iter, eps, &iters, &sse);
    clock_t t1 = clock();
    double ms = 1000.0 * (double)(t1 - t0) / (double)CLOCKS_PER_SEC;

    printf("K-means 1D (naive)\n");
    printf("N=%d K=%d max_iter=%d eps=%g\n", N, K, max_iter, eps);
    printf("Iterações: %d | SSE final: %.6f | Tempo: %.1f ms\n", iters, sse, ms);

    write_assign_csv(outAssign, assign, N);
    write_centroids_csv(outCentroid, C, K);

    free(assign); free(X); free(C);
    return 0;
}


Overwriting kmeans_1d_naive.c


In [30]:
%%shell

gcc -O2 -std=c99 kmeans_1d_naive.c -o kmeans_1d_naive -lm
./kmeans_1d_naive dados.csv centroides_iniciais.csv 50 0.000001 assign.csv centroids.csv
cat centroids.csv



K-means 1D (naive)
N=20 K=4 max_iter=50 eps=1e-06
Iterações: 4 | SSE final: 344.875000 | Tempo: 0.0 ms
6.300000
20.375000
2.900000
112.500000


In [31]:
%%writefile kmeans_omp.c

#include <stdio.h>
#include <stdlib.h>
#include <omp.h>
#include <string.h>
#include <float.h>

static double etapa_atribuir(const double *pontos, const double *centros, int *grupo,
                             int N, int K)
{
    double erro = 0.0;

    #pragma omp parallel for reduction(+:erro)
    for(int i=0; i<N; i++){
        int melhor = -1;
        double melhord = DBL_MAX;

        for(int c=0; c<K; c++){
            double dif = pontos[i] - centros[c];
            double d = dif*dif;
            if(d < melhord) { melhord = d; melhor = c; }
        }
        grupo[i] = melhor;
        erro += melhord;
    }
    return erro;
}

static void etapa_atualizar(const double *pontos, double *centros, const int *grupo, int N, int K)
{
    double *soma = (double*) calloc((size_t)K, sizeof(double));
    int *cont = (int*) calloc((size_t)K, sizeof(int));

    #pragma omp parallel for reduction(+:soma[:K], cont[:K])
    for(int i=0; i<N; i++){
        int a = grupo[i];
        cont[a] += 1;
        soma[a] += pontos[i];
    }

    for(int c=0; c<K; c++) {
        if(cont[c] > 0) centros[c] = soma[c] / (double)cont[c];
        else centros[c] = pontos[0];
    }

    free(soma); free(cont);
}

int main() {
    int N = 1000000;
    int K = 5;

    double *pontos = (double*)malloc(N * sizeof(double));
    double *centros = (double*)malloc(K * sizeof(double));
    int *grupo = (int*)malloc(N * sizeof(int));

    for(int i=0; i<N; i++) pontos[i] = (double)i;
    for(int c=0; c<K; c++) centros[c] = (double)(c * N / K);

    printf("Executando K-Means (OpenMP) com N=%d, K=%d\n", N, K);
    double t_inicio = omp_get_wtime();

    double erro = etapa_atribuir(pontos, centros, grupo, N, K);
    etapa_atualizar(pontos, centros, grupo, N, K);

    double t_fim = omp_get_wtime();
    printf("SSE (Erro): %f\n", erro);
    printf("Novos centros (exemplo 0): %f\n", centros[0]);
    printf("Tempo: %f segundos\n", t_fim - t_inicio);

    free(pontos);
    free(centros);
    free(grupo);
    return 0;
}

Overwriting kmeans_omp.c


In [32]:
!gcc -o kmeans_omp kmeans_omp.c -fopenmp -O3 -lm
!./kmeans_omp

Executando K-Means (OpenMP) com N=1000000, K=5
SSE (Erro): 5333313333500000.000000
Novos centros (exemplo 0): 50000.000000
Tempo: 0.007617 segundos


In [33]:
%%writefile kmeans_cuda.cu

#include <stdio.h>
#include <stdlib.h>
#include <float.h>
#include <omp.h>
#include <string.h>

// Macro para verificação de erros CUDA
#define CUDA_CHECK(err) { \
    if(err != cudaSuccess) { \
        fprintf(stderr, "Erro CUDA na linha %d: %s\n", __LINE__, cudaGetErrorString(err)); \
        exit(1); \
    } \
}

__global__
void atribuir_kernel(const float *pontos, const float *centros, int *grupo_gpu,
                     float *erros_gpu, int N, int K)
{
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i >= N) return;

    int melhor = -1;
    float melhord = FLT_MAX;

    for(int c=0; c<K; c++){
        float dif = pontos[i] - centros[c];
        float d = dif*dif;
        if(d < melhord) { melhord = d; melhor = c; }
    }

    grupo_gpu[i] = melhor;
    erros_gpu[i] = melhord;
}

float atribuir_cuda_host(const float *pontos_gpu, const float *centros_gpu, int *grupo_gpu,
                         float *erros_gpu, float *erros_cpu, int N, int K)
{
    int TAM_BLOCO = 256;
    int NUM_BLOCOS = (N + TAM_BLOCO - 1) / TAM_BLOCO;

    atribuir_kernel<<<NUM_BLOCOS, TAM_BLOCO>>>(pontos_gpu, centros_gpu, grupo_gpu, erros_gpu, N, K);

    CUDA_CHECK(cudaGetLastError()); // Verifica se o kernel foi lançado corretamente
    CUDA_CHECK(cudaDeviceSynchronize()); // Espera o kernel terminar

    CUDA_CHECK(cudaMemcpy(erros_cpu, erros_gpu, N * sizeof(float), cudaMemcpyDeviceToHost));

    float erro_total = 0.0f;
    #pragma omp parallel for reduction(+:erro_total)
    for (int i = 0; i < N; i++) {
        erro_total += erros_cpu[i];
    }
    return erro_total;
}

static void etapa_atualizar_cpu(const float *pontos, float *centros, const int *grupo, int N, int K)
{
    double *soma = (double*) calloc((size_t)K, sizeof(double));
    int *cont = (int*) calloc((size_t)K, sizeof(int));

    #pragma omp parallel for reduction(+:soma[:K], cont[:K])
    for(int i=0; i<N; i++){
        int a = grupo[i];
        cont[a] += 1;
        soma[a] += (double)pontos[i]; // Converte para double para somar
    }

    for(int c=0; c<K; c++) {
        if(cont[c] > 0) centros[c] = (float)(soma[c] / (double)cont[c]);
        else centros[c] = pontos[0];
    }
    free(soma); free(cont);
}

int main() {
    int N = 1000000;
    int K = 5;

    size_t bytes_pontos = N * sizeof(float);
    size_t bytes_grupos = N * sizeof(int);
    size_t bytes_erros = N * sizeof(float);
    size_t bytes_centros = K * sizeof(float);

    float *h_pontos = (float*)malloc(bytes_pontos);
    float *h_centros = (float*)malloc(bytes_centros);
    int *h_grupo = (int*)malloc(bytes_grupos);
    float *h_erros_cpu = (float*)malloc(bytes_erros);

    for(int i=0; i<N; i++) h_pontos[i] = (float)i;
    for(int c=0; c<K; c++) h_centros[c] = (float)(c * N / K);

    float *d_pontos, *d_centros, *d_erros;
    int *d_grupo;
    CUDA_CHECK(cudaMalloc(&d_pontos, bytes_pontos));
    CUDA_CHECK(cudaMalloc(&d_centros, bytes_centros));
    CUDA_CHECK(cudaMalloc(&d_grupo, bytes_grupos));
    CUDA_CHECK(cudaMalloc(&d_erros, bytes_erros));

    printf("Executando K-Means (CUDA) com N=%d, K=%d\n", N, K);

    cudaEvent_t inicio, fim, inicio_kernel, fim_kernel, inicio_d2h, fim_d2h;
    CUDA_CHECK(cudaEventCreate(&inicio));       CUDA_CHECK(cudaEventCreate(&fim));
    CUDA_CHECK(cudaEventCreate(&inicio_kernel)); CUDA_CHECK(cudaEventCreate(&fim_kernel));
    CUDA_CHECK(cudaEventCreate(&inicio_d2h));    CUDA_CHECK(cudaEventCreate(&fim_d2h));

    CUDA_CHECK(cudaEventRecord(inicio));
    CUDA_CHECK(cudaMemcpy(d_pontos, h_pontos, bytes_pontos, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_centros, h_centros, bytes_centros, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaEventRecord(fim));
    CUDA_CHECK(cudaEventSynchronize(fim));
    float tempo_h2d;
    CUDA_CHECK(cudaEventElapsedTime(&tempo_h2d, inicio, fim));

    CUDA_CHECK(cudaEventRecord(inicio_kernel));
    float erro = atribuir_cuda_host(d_pontos, d_centros, d_grupo, d_erros, h_erros_cpu, N, K);
    CUDA_CHECK(cudaEventRecord(fim_kernel));
    CUDA_CHECK(cudaEventSynchronize(fim_kernel));
    float tempo_kernel;
    CUDA_CHECK(cudaEventElapsedTime(&tempo_kernel, inicio_kernel, fim_kernel));

    CUDA_CHECK(cudaEventRecord(inicio_d2h));
    CUDA_CHECK(cudaMemcpy(h_grupo, d_grupo, bytes_grupos, cudaMemcpyDeviceToHost));
    CUDA_CHECK(cudaEventRecord(fim_d2h));
    CUDA_CHECK(cudaEventSynchronize(fim_d2h));
    float tempo_d2h;
    CUDA_CHECK(cudaEventElapsedTime(&tempo_d2h, inicio_d2h, fim_d2h));

    etapa_atualizar_cpu(h_pontos, h_centros, h_grupo, N, K);

    float tempo_total_gpu = tempo_h2d + tempo_kernel + tempo_d2h;

    printf("SSE (Erro): %f\n", erro);
    printf("Novos centros (exemplo 0): %f\n", h_centros[0]);
    printf("Tempo H2D: %.4f ms\n", tempo_h2d);
    printf("Tempo Kernel: %.4f ms\n", tempo_kernel);
    printf("Tempo D2H: %.4f ms\n", tempo_d2h);
    printf("Tempo Total GPU: %.4f ms (%.6f s)\n", tempo_total_gpu, tempo_total_gpu / 1000.0);

    free(h_pontos); free(h_centros); free(h_grupo); free(h_erros_cpu);
    cudaFree(d_pontos); cudaFree(d_centros); cudaFree(d_grupo); cudaFree(d_erros);
    cudaEventDestroy(inicio); cudaEventDestroy(fim);

    return 0;
}

Overwriting kmeans_cuda.cu


In [34]:
!nvcc -o kmeans_cuda kmeans_cuda.cu -O3 -Xcompiler -fopenmp -lgomp -arch=sm_75
!./kmeans_cuda

Executando K-Means (CUDA) com N=1000000, K=5
SSE (Erro): 5332958885969920.000000
Novos centros (exemplo 0): 50000.000000
Tempo H2D: 1.0149 ms
Tempo Kernel: 3.6001 ms
Tempo D2H: 2.8229 ms
Tempo Total GPU: 7.4379 ms (0.007438 s)


In [35]:
%%writefile kmeans_mpi.c

#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <float.h>
#include <mpi.h>

static double etapa_atribuir(const double *pontos, const double *centros, int *grupo,
                             int N, int K)
{
    double erro = 0.0;
    for(int i=0; i<N; i++){
        int melhor = -1;
        double melhord = DBL_MAX;

        for(int c=0; c<K; c++){
            double dif = pontos[i] - centros[c];
            double d = dif*dif;
            if(d < melhord) { melhord = d; melhor = c; }
        }
        grupo[i] = melhor;
        erro += melhord;
    }
    return erro;
}

int main(int argc, char **argv) {

    MPI_Init(&argc, &argv);
    int rank, P;
    MPI_Comm_rank(MPI_COMM_WORLD, &rank);
    MPI_Comm_size(MPI_COMM_WORLD, &P);

    int N_global = 1000000;
    int K = 5;

    int N_local = N_global / P;
    int inicio_local = rank * N_local;
    if (rank == P - 1) {
        N_local = N_global - (rank * N_local);
    }

    double *pontos_local = (double*)malloc(N_local * sizeof(double));
    int *grupo_local = (int*)malloc(N_local * sizeof(int));
    double *centros = (double*)malloc(K * sizeof(double));

    for(int i=0; i<N_local; i++) {
        pontos_local[i] = (double)(inicio_local + i);
    }
    if (rank == 0) {
        for(int c=0; c<K; c++) centros[c] = (double)(c * N_global / K);
    }

    MPI_Bcast(centros, K, MPI_DOUBLE, 0, MPI_COMM_WORLD);

    double *soma_local = (double*) calloc(K, sizeof(double));
    int *cont_local = (int*) calloc(K, sizeof(int));
    double *soma_global = (double*) calloc(K, sizeof(double));
    int *cont_global = (int*) calloc(K, sizeof(int));

    if (rank == 0) {
        printf("Executando K-Means (MPI) com N=%d, K=%d em P=%d processos\n", N_global, K, P);
    }

    double t_inicio = MPI_Wtime();

    double erro_local = etapa_atribuir(pontos_local, centros, grupo_local, N_local, K);

    memset(soma_local, 0, K * sizeof(double));
    memset(cont_local, 0, K * sizeof(int));

    for(int i=0; i<N_local; i++){
        int a = grupo_local[i];
        cont_local[a] += 1;
        soma_local[a] += pontos_local[i];
    }

    double erro_global;
    MPI_Reduce(&erro_local, &erro_global, 1, MPI_DOUBLE, MPI_SUM, 0, MPI_COMM_WORLD);

    MPI_Allreduce(soma_local, soma_global, K, MPI_DOUBLE, MPI_SUM, MPI_COMM_WORLD);
    MPI_Allreduce(cont_local, cont_global, K, MPI_INT, MPI_SUM, MPI_COMM_WORLD);

    for(int c=0; c<K; c++) {
        if(cont_global[c] > 0) centros[c] = soma_global[c] / (double)cont_global[c];
        else centros[c] = pontos_local[0];
    }

    double t_fim = MPI_Wtime();

    if (rank == 0) {
        printf("SSE (Erro Global): %f\n", erro_global);
        printf("Novos centros (exemplo 0): %f\n", centros[0]);
        printf("Tempo: %f segundos\n", t_fim - t_inicio);
    }

    free(pontos_local);
    free(grupo_local);
    free(centros);
    free(soma_local);
    free(cont_local);
    free(soma_global);
    free(cont_global);

    MPI_Finalize();
    return 0;
}

Overwriting kmeans_mpi.c


In [36]:
%%shell

mpicc -o kmeans_mpi kmeans_mpi.c -O3 -lm

echo "--- MPI Results ---"
echo "Processes | Time (s)"
echo "----------|----------"
for procs in 1 2 4 8; do
    echo -n "$procs       | "
    mpirun --allow-run-as-root --oversubscribe -np $procs ./kmeans_mpi | grep "Tempo:" | awk '{print $2}'
done

--- MPI Results ---
Processes | Time (s)
----------|----------
1       | 0.007874
2       | 0.008698
4       | 0.013908
8       | 0.012786


In [38]:
!OMP_NUM_THREADS=1 ./kmeans_omp

Executando K-Means (OpenMP) com N=1000000, K=5
SSE (Erro): 5333313333500000.000000
Novos centros (exemplo 0): 50000.000000
Tempo: 0.014688 segundos
